# Week 8 — Monday Lab Notebook  
## Machine Learning Foundations + Workflow

### Today’s Goal

By the end of today, students should be able to:

- understand what machine learning means in simple words
- explain the difference between rule-based systems and machine learning systems
- identify **features** and **labels** in a dataset
- understand why we split data into **train**, **validation**, and **test** sets
- understand what **data leakage** means and why it is dangerous
- understand the basic idea of **evaluation metrics**
- understand what a **baseline model** is and why it matters
- follow the simple workflow of a machine learning project from data to evaluation

This notebook is for **lab teaching**. It uses easy wording and small examples first.


## 3-Hour Structure

**0:00–0:20** What is machine learning? ML vs rule-based systems  
**0:20–0:45** Build a small dataset and understand features + labels  
**0:45–1:15** Train / validation / test split  
**1:15–1:40** Data leakage in simple wording  
**1:40–2:10** Metrics idea: accuracy, confusion matrix, MAE  
**2:10–2:40** Baseline models  
**2:40–3:00** Full workflow recap + mini class practice + tasks


## Part A — What is Machine Learning?

Machine learning means:

> we give data to a computer, and the computer learns a pattern from that data.

Then the computer uses that pattern to make predictions on new data.

### Simple examples

- predict if a student may pass or fail
- predict house price
- detect spam emails
- recommend videos or products
- predict whether a customer may leave a business

Machine learning is useful when writing fixed rules for every case becomes difficult.


## Part B — Rule-Based System vs Machine Learning

A **rule-based system** works by fixed conditions written by a human.

Example:

- if marks are 90 or above, grade is A
- if marks are 80 to 89, grade is B
- if marks are below 50, fail

This works well when rules are very clear.

But many real problems are not this simple.

For example, to predict whether a student will pass, we may need to consider:

- hours studied
- attendance
- assignments submitted
- previous marks
- quiz performance
- maybe even more factors together

That is where machine learning becomes helpful.


### Small idea

- **Rule-based:** human writes the logic directly  
- **Machine learning:** machine learns the logic from examples


## Part C — A Small Rule-Based Example in Python

This is not machine learning. This is just manual logic.


In [ ]:
def rule_based_pass_prediction(hours_studied, attendance):
    if hours_studied >= 4 and attendance >= 80:
        return "Likely Pass"
    else:
        return "Likely Fail"

print(rule_based_pass_prediction(5, 85))
print(rule_based_pass_prediction(2, 70))
print(rule_based_pass_prediction(4, 60))


### Why is this limited?

Because real life is not always that neat.

A student with:

- low attendance but very high previous marks
- medium study hours but strong assignments
- weak quiz marks but better final preparation

may still pass.

A fixed rule may miss many such cases.


## Part D — Import Libraries

We will use:

- `pandas` for tables
- `numpy` for numeric work
- `scikit-learn` for basic machine learning utilities and baseline models


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

print("Libraries imported successfully.")


## Part E — Create a Small Student Dataset

We will use a simple dataset of students.

Columns:

- `hours_studied`
- `attendance`
- `assignments_submitted`
- `previous_marks`
- `final_score`
- `pass`

Here:

- `final_score` is a **numeric target** for a regression example
- `pass` is a **yes/no target** for a classification example


In [ ]:
students = pd.DataFrame({
    "hours_studied": [1, 2, 3, 4, 5, 6, 2, 7, 3, 8, 4, 5, 6, 1, 2, 7, 8, 3, 4, 6],
    "attendance": [55, 60, 65, 70, 75, 80, 58, 88, 62, 95, 72, 78, 84, 50, 57, 90, 96, 64, 73, 85],
    "assignments_submitted": [2, 3, 4, 5, 6, 7, 3, 8, 4, 9, 5, 6, 7, 1, 2, 8, 9, 4, 5, 7],
    "previous_marks": [40, 45, 50, 55, 60, 68, 44, 74, 52, 85, 58, 63, 71, 38, 42, 78, 88, 49, 57, 73],
    "final_score": [42, 48, 54, 59, 64, 71, 47, 79, 55, 90, 61, 67, 75, 39, 44, 82, 92, 53, 60, 76]
})

students["pass"] = np.where(students["final_score"] >= 50, 1, 0)

students


## Part F — Look at the Data First

Before doing machine learning, always inspect the data.


In [ ]:
print(students.head())
print()
print(students.info())


In [ ]:
students.describe()


### Why do we inspect first?

Because we want to know:

- how many rows we have
- what each column means
- which columns are numeric
- whether there are missing values
- whether the target column looks reasonable


## Part G — What are Features and Labels?

In machine learning:

- **Features** are the input columns used to make predictions
- **Label** or **target** is the output we want to predict

### Example 1: Predict pass or fail

Features could be:

- `hours_studied`
- `attendance`
- `assignments_submitted`
- `previous_marks`

Label could be:

- `pass`

### Example 2: Predict final score

Features could be the same.

Label could be:

- `final_score`


In [ ]:
feature_columns = ["hours_studied", "attendance", "assignments_submitted", "previous_marks"]

X = students[feature_columns]
y_class = students["pass"]
y_reg = students["final_score"]

print("Features (X):")
print(X.head())
print()
print("Classification label (y_class):")
print(y_class.head())
print()
print("Regression label (y_reg):")
print(y_reg.head())


### Very important

`X` usually means input features.  
`y` usually means target or label.

This naming style is very common in machine learning.


## Part H — One Input, Many Inputs

A model can use:

- **one feature** only
- or **many features** together

In most real problems, many features work better than only one feature.


In [ ]:
X_one_feature = students[["hours_studied"]]
X_many_features = students[["hours_studied", "attendance", "assignments_submitted", "previous_marks"]]

print("One feature shape:", X_one_feature.shape)
print("Many features shape:", X_many_features.shape)


### Shape meaning

If shape is `(20, 4)`, it means:

- 20 rows
- 4 feature columns


## Part I — Why We Cannot Test on the Same Data We Trained On

Suppose a student memorizes past paper questions only.

Then in the exam, if the exact same questions come again, the student may score high.

But that does not prove deep understanding.

Machine learning is similar.

If we train and test on the same data, the score may look very good, but it may not work well on new unseen data.

That is why we split the data.


## Part J — Train, Validation, and Test Sets

### Train set
Used to teach the model.

### Validation set
Used to check model quality while developing.
We may compare models or settings on this set.

### Test set
Used at the end for final evaluation.
This should behave like unseen real-world data.

### Simple memory trick

- **train** = learn  
- **validation** = check during development  
- **test** = final exam


## Part K — First Split: Train and Temporary Set

We will first split the data into:

- training data
- temporary data

Then we will split the temporary data into:

- validation data
- test data


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_class, test_size=0.30, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_temp shape:", X_temp.shape)
print("y_train shape:", y_train.shape)
print("y_temp shape:", y_temp.shape)


## Part L — Second Split: Validation and Test


In [ ]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)


### What happened here?

If the full data has 20 rows, then roughly:

- 14 rows went to train
- 3 rows went to validation
- 3 rows went to test

Because this dataset is very small, the exact numbers are also very small.

In real projects, datasets are usually much larger.


## Part M — See the Actual Rows in Each Split

This helps students understand that the rows are actually being separated.


In [ ]:
print("Train rows:")
print(X_train)
print()
print("Validation rows:")
print(X_val)
print()
print("Test rows:")
print(X_test)


## Part N — Why Do We Use `random_state`?

`random_state` gives reproducible splitting.

That means when we run the notebook again, we get the same split.

This is useful for:

- teaching
- debugging
- comparing models fairly


In [ ]:
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X, y_class, test_size=0.25, random_state=42
)

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X, y_class, test_size=0.25, random_state=42
)

print("Are both test sets equal?", X_test_a.equals(X_test_b))


## Part O — Classification Problem vs Regression Problem

### Classification
When the output belongs to a class or category.

Examples:

- pass / fail
- spam / not spam
- disease / no disease

### Regression
When the output is a numeric value.

Examples:

- house price
- exam score
- temperature
- sales amount


## Part P — Our Dataset Has Both Examples

- `pass` is for **classification**
- `final_score` is for **regression**


In [ ]:
print("Unique values in pass column:", students["pass"].unique())
print("Sample values from final_score:", students["final_score"].head().tolist())


## Part Q — A Tiny Classification Model for Demo

Today the focus is not advanced models.
We just want to see the workflow.

So we will use a very small decision tree.


In [ ]:
clf = DecisionTreeClassifier(max_depth=2, random_state=42)
clf.fit(X_train, y_train)

val_predictions = clf.predict(X_val)
test_predictions = clf.predict(X_test)

print("Validation predictions:", val_predictions)
print("Test predictions:", test_predictions)


## Part R — A Tiny Regression Model for Demo

Now we will do the same idea for numeric prediction.


In [ ]:
X_train_r, X_temp_r, y_train_r, y_temp_r = train_test_split(
    X, y_reg, test_size=0.30, random_state=42
)

X_val_r, X_test_r, y_val_r, y_test_r = train_test_split(
    X_temp_r, y_temp_r, test_size=0.50, random_state=42
)

reg = DecisionTreeRegressor(max_depth=2, random_state=42)
reg.fit(X_train_r, y_train_r)

val_pred_r = reg.predict(X_val_r)
test_pred_r = reg.predict(X_test_r)

print("Validation predictions:", val_pred_r)
print("Test predictions:", test_pred_r)


### Important note

This is only a demo model.

Later weeks will cover better models in more detail.
Today we are learning the **workflow**, not trying to build the best model.


## Part S — What is Data Leakage?

Data leakage means:

> the model gets information that it should not have during training.

This creates unrealistically high performance.

Then later, on real new data, the model may fail badly.

Leakage is one of the most common beginner mistakes.


### Simple real-life meaning

It is like giving exam answers to a student before the exam, then saying:

> wow, the student is amazing.

That result is not fair.


## Part T — Leakage Example 1: Using a Target-Derived Column

Suppose we want to predict `pass`.

But `pass` was made from `final_score`.

If we include `final_score` inside the features, then the model gets a clue that is almost the answer itself.

That is leakage.


In [ ]:
X_leaky = students[["hours_studied", "attendance", "assignments_submitted", "previous_marks", "final_score"]]
y_leaky = students["pass"]

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky, y_leaky, test_size=0.30, random_state=42
)

leaky_model = DecisionTreeClassifier(max_depth=2, random_state=42)
leaky_model.fit(X_train_leaky, y_train_leaky)
leaky_predictions = leaky_model.predict(X_test_leaky)

print("Leaky model accuracy:", accuracy_score(y_test_leaky, leaky_predictions))


### Why is that wrong?

Because `final_score` is strongly connected to `pass`.

So the model is not truly learning from earlier information.
It is using information that is too close to the answer.


## Part U — Correct Feature Set Without Leakage


In [ ]:
X_clean = students[["hours_studied", "attendance", "assignments_submitted", "previous_marks"]]
y_clean = students["pass"]

X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_clean, y_clean, test_size=0.30, random_state=42
)

clean_model = DecisionTreeClassifier(max_depth=2, random_state=42)
clean_model.fit(X_train_clean, y_train_clean)
clean_predictions = clean_model.predict(X_test_clean)

print("Clean model accuracy:", accuracy_score(y_test_clean, clean_predictions))


## Part V — Leakage Example 2: Looking at Test Data Too Early

Another leakage mistake is this:

- we try many model settings
- we keep checking the test set again and again
- then we choose the setting that works best on test

Now the test set is no longer a fair final exam.

That is why we should mainly use the **validation set** during development and keep the **test set** for the end.


## Part W — What are Metrics?

A metric is a number that tells us how good or bad a model is.

Different problems use different metrics.


### Common examples

#### For classification

- accuracy
- precision
- recall
- F1-score
- ROC-AUC

#### For regression

- MAE
- MSE
- RMSE
- R²

Today we will only focus on the basic idea of **accuracy** and **MAE**.


## Part X — Accuracy in Simple Words

Accuracy means:

> out of all predictions, how many were correct?

Formula idea:

accuracy = correct predictions / total predictions


In [ ]:
val_accuracy = accuracy_score(y_val, val_predictions)
test_accuracy = accuracy_score(y_test, test_predictions)

print("Validation accuracy:", val_accuracy)
print("Test accuracy:", test_accuracy)


## Part Y — Confusion Matrix in Simple Words

A confusion matrix helps us see:

- correct positives
- correct negatives
- wrong positives
- wrong negatives

For a beginner, the main purpose is this:

> it shows not just how many were correct, but also what kind of mistakes happened.


In [ ]:
cm = confusion_matrix(y_test, test_predictions)
print(cm)


### Reading a 2×2 confusion matrix

For binary classification, the positions usually mean:

- top-left = actual 0, predicted 0
- top-right = actual 0, predicted 1
- bottom-left = actual 1, predicted 0
- bottom-right = actual 1, predicted 1

So it helps us know where the model is making mistakes.


## Part Z — MAE in Simple Words

MAE means **Mean Absolute Error**.

It is used in regression.

It tells us the average size of prediction error.

Smaller MAE is better.


In [ ]:
val_mae = mean_absolute_error(y_val_r, val_pred_r)
test_mae = mean_absolute_error(y_test_r, test_pred_r)

print("Validation MAE:", val_mae)
print("Test MAE:", test_mae)


### Example meaning

If MAE is about 4, it means the model is wrong by about 4 marks on average.

That is a simple beginner way to understand it.


## Part AA — Why One Metric is Not Always Enough

A model may have decent accuracy but still be bad in some cases.

For example, in a class of 100 students:

- 95 students pass
- 5 students fail

If a weak model predicts **everyone will pass**, accuracy becomes 95%.

That sounds good, but it completely misses all failing students.

So we must think carefully about the problem before choosing a metric.


## Part AB — What is a Baseline Model?

A baseline model is a very simple starting model.

We use it to answer this question:

> is my smarter model really better than a very basic approach?

Without a baseline, a model result has less meaning.


### Examples of simple baselines

- always predict the most common class
- always predict the average value
- use a very small simple model first

If your advanced model cannot beat the baseline, that is a warning sign.


## Part AC — Baseline Classification Model

We will use `DummyClassifier`.

This model does not learn a real pattern.
It just uses a very simple strategy.


In [ ]:
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_train)

dummy_val_pred = dummy_clf.predict(X_val)
dummy_test_pred = dummy_clf.predict(X_test)

print("Dummy validation accuracy:", accuracy_score(y_val, dummy_val_pred))
print("Dummy test accuracy:", accuracy_score(y_test, dummy_test_pred))


## Part AD — Compare Baseline vs Simple Model


In [ ]:
comparison_classification = pd.DataFrame({
    "model": ["DummyClassifier", "DecisionTreeClassifier"],
    "validation_accuracy": [accuracy_score(y_val, dummy_val_pred), accuracy_score(y_val, val_predictions)],
    "test_accuracy": [accuracy_score(y_test, dummy_test_pred), accuracy_score(y_test, test_predictions)]
})

comparison_classification


### What should students notice?

If the decision tree is better than the dummy model, that means it has learned something useful.

If it is not better, then the model may not be helping much.


## Part AE — Baseline Regression Model

Now we do the same idea for numeric prediction.

`DummyRegressor` with strategy `mean` predicts the average training target.


In [ ]:
dummy_reg = DummyRegressor(strategy="mean")
dummy_reg.fit(X_train_r, y_train_r)

dummy_val_pred_r = dummy_reg.predict(X_val_r)
dummy_test_pred_r = dummy_reg.predict(X_test_r)

print("Dummy validation MAE:", mean_absolute_error(y_val_r, dummy_val_pred_r))
print("Dummy test MAE:", mean_absolute_error(y_test_r, dummy_test_pred_r))


## Part AF — Compare Regression Baseline vs Simple Model


In [ ]:
comparison_regression = pd.DataFrame({
    "model": ["DummyRegressor", "DecisionTreeRegressor"],
    "validation_MAE": [mean_absolute_error(y_val_r, dummy_val_pred_r), mean_absolute_error(y_val_r, val_pred_r)],
    "test_MAE": [mean_absolute_error(y_test_r, dummy_test_pred_r), mean_absolute_error(y_test_r, test_pred_r)]
})

comparison_regression


### For MAE, smaller is better

So when comparing regression models:

- lower MAE = better
- higher MAE = worse


## Part AG — Full Machine Learning Workflow in Simple Steps

A beginner-friendly workflow looks like this:

1. understand the problem  
2. collect or load data  
3. inspect and clean the data  
4. choose features and label  
5. split data into train, validation, and test  
6. train a simple model  
7. evaluate using metrics  
8. compare with a baseline  
9. improve the model later  
10. keep the test set for final checking


## Part AH — One Mini End-to-End Example

Let us write one compact workflow for classification.


In [ ]:
# 1. choose features and target
X_demo = students[["hours_studied", "attendance", "assignments_submitted", "previous_marks"]]
y_demo = students["pass"]

# 2. split data
X_train_demo, X_test_demo, y_train_demo, y_test_demo = train_test_split(
    X_demo, y_demo, test_size=0.30, random_state=42
)

# 3. train a simple model
model_demo = DecisionTreeClassifier(max_depth=2, random_state=42)
model_demo.fit(X_train_demo, y_train_demo)

# 4. predict
pred_demo = model_demo.predict(X_test_demo)

# 5. evaluate
acc_demo = accuracy_score(y_test_demo, pred_demo)
cm_demo = confusion_matrix(y_test_demo, pred_demo)

print("Accuracy:", acc_demo)
print("Confusion Matrix:
", cm_demo)


## Part AI — Common Beginner Mistakes

1. using the label column inside features  
2. checking the test set again and again during development  
3. not understanding what the target actually means  
4. trusting accuracy only  
5. not comparing with a baseline  
6. training on the full data before fair evaluation  
7. thinking a high score always means a useful model


## Part AJ — Mini Real-Life Example 1: Predict Student Pass/Fail

Problem:

A college wants to identify students who may need support before the final exam.

Possible features:

- attendance
- assignments submitted
- previous marks
- study hours

Possible label:

- pass or fail

Why useful?

Because teachers can help weak students earlier.


## Part AK — Mini Real-Life Example 2: Predict House Price

Problem:

A property business wants to estimate house prices.

Possible features:

- house size
- number of rooms
- location score
- age of house

Possible label:

- price

This is a **regression** problem.


## Part AL — Mini Real-Life Example 3: Email Spam Detection

Problem:

An email system wants to decide whether a message is spam.

Possible features:

- number of links
- suspicious words
- sender history
- message length

Possible label:

- spam or not spam

This is a **classification** problem.


## Part AM — Small Interpretation Practice 1

A model gets 92% accuracy.

Question:

Can we say the model is definitely excellent?

Answer idea:

Not always.
We still need to check:

- what kind of data it used
- whether there was leakage
- whether classes are balanced
- which mistakes it makes
- whether it beats the baseline


## Part AN — Small Interpretation Practice 2

A model has test accuracy lower than training accuracy.

Is that always bad?

Not necessarily.

Usually training accuracy is a bit higher.
What matters is whether the gap is too large.

A very large gap may suggest overfitting.


## Part AO — Small Interpretation Practice 3

A dummy model gets 80% accuracy.
Your model gets 81% accuracy.

Question:

Is your model impressive?

Answer idea:

Not really.
It is only slightly better.
We may need better features, better evaluation, or a better model.


## Part AP — Mini Practice in Class

Try these before moving to the after-lab tasks:

1. identify the features and label in the student dataset  
2. create `X` and `y` for predicting `pass`  
3. split the data into train and test  
4. train a `DummyClassifier`  
5. train a small `DecisionTreeClassifier`  
6. compare both accuracies  
7. repeat for regression using `final_score`  
8. compute MAE for the regression model  
9. create one leakage example and explain why it is wrong  
10. explain in your own words why test data should be kept for the end


## Part AQ — Recap

Today we learned how to:

- understand ML in simple words
- compare ML with rule-based systems
- identify features and labels
- split data into train, validation, and test
- understand data leakage
- understand basic metrics like accuracy and MAE
- compare simple models with baselines
- follow the basic machine learning workflow

This foundation is very important because later machine learning topics make more sense only when the workflow is clear.


# After-Lab Tasks (10)

### Task 1
Create a small DataFrame with at least 12 rows and these columns: `study_hours`, `attendance`, `assignment_score`, and `result`.

### Task 2
From your DataFrame, clearly write which columns are **features** and which column is the **label**.

### Task 3
Create `X` and `y` for a **classification** task.

### Task 4
Split your data into `train` and `test` sets using `train_test_split()`.

### Task 5
Train a `DummyClassifier` on your training data and calculate its accuracy on the test data.

### Task 6
Train a small `DecisionTreeClassifier` and compare its accuracy with the dummy model.

### Task 7
Create a numeric target column such as `final_marks` and turn your dataset into a **regression** example.

### Task 8
Train a `DummyRegressor` and a `DecisionTreeRegressor`, then compare their MAE values.

### Task 9
Make one example of **data leakage** in your dataset and write 3 to 4 lines explaining why it is leakage.

### Task 10
Write the full machine learning workflow in your own words in 8 to 10 lines, from loading data up to final evaluation.


# Optional Homework Challenge

Create your own small dataset for a real-life problem such as:

- student pass prediction
- customer churn prediction
- house price prediction
- employee performance prediction

Then do all of the following:

1. explain the problem in 2 to 3 lines  
2. identify features and target  
3. state whether it is classification or regression  
4. split the data into train and test  
5. build one baseline model  
6. build one simple tree-based model  
7. evaluate using one suitable metric  
8. explain whether your real model beats the baseline  
9. create one example of a wrong leaky feature and explain it  
10. write a final conclusion in 6 to 8 lines
